# Capstone 2
For this project, I will be working through reading, cleaning, and analyzing data for a fictional business using sample data. This project will call on my ability to read data and extract useful information using my Python skills.

The Sales Manager assigned to me is Miami Vue.

In [2]:
import pandas as pd
import numpy as np
given_sales_manager = "Miami Vue"

In [3]:
product_categories = pd.read_csv("ProductCategories.csv")
print(product_categories.info())
print(product_categories.head(1))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52 entries, 0 to 51
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   CategoryID     52 non-null     int64 
 1   Category       52 non-null     object
 2   SubcategoryID  52 non-null     object
 3   Subcategory    52 non-null     object
dtypes: int64(1), object(3)
memory usage: 1.8+ KB
None
   CategoryID                  Category SubcategoryID Subcategory
0         120  Technology & Accessories       120-tab     Tablets


In [4]:
products = pd.read_csv("Products.csv")
print(products.info())
print(products.head(1))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 669 entries, 0 to 668
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Prod Num       669 non-null    object
 1   Product        669 non-null    object
 2   CategoryID     669 non-null    int64 
 3   SubcategoryID  669 non-null    object
dtypes: int64(1), object(3)
memory usage: 21.0+ KB
None
    Prod Num           Product  CategoryID SubcategoryID
0  105248-IT  TCL NXTPAPER 10s         120       120-tab


In [5]:
store_details = pd.read_csv("StoreDetail.csv")
print(store_details.info())
print(store_details.head(1))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 111 entries, 0 to 110
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Store Location     111 non-null    object
 1   State              111 non-null    object
 2   Store ID           111 non-null    int64 
 3   Territory Manager  111 non-null    object
 4   Region             111 non-null    object
 5   Region Director    111 non-null    object
dtypes: int64(1), object(5)
memory usage: 5.3+ KB
None
  Store Location     State  Store ID Territory Manager Region  Region Director
0         Aurora  Colorado       701          Jim Heck   West  Cassie Chambers


In [6]:
store_sales = pd.read_csv("StoreSales.csv")
print(store_sales.info())
print(store_sales.head(1))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 335129 entries, 0 to 335128
Data columns (total 5 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Transaction Date  335129 non-null  object 
 1   Store ID          335129 non-null  int64  
 2   RewardsID         34943 non-null   float64
 3   Prod Num          335129 non-null  object 
 4   Sale Amount       335129 non-null  float64
dtypes: float64(2), int64(1), object(2)
memory usage: 12.8+ MB
None
  Transaction Date  Store ID  RewardsID  Prod Num  Sale Amount
0         1/1/2022       702        NaN  105349-M          8.0


In [7]:
customer_list = pd.read_csv("customer_list.csv")
print(customer_list.info())
print(customer_list.head(1))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 521 entries, 0 to 520
Data columns (total 1 columns):
 #   Column                                           Non-Null Count  Dtype 
---  ------                                           --------------  ----- 
 0   cust_id|date|time|name|email|phone|sms-opt-out   521 non-null    object
dtypes: object(1)
memory usage: 4.2+ KB
None
     cust_id|date|time|name|email|phone|sms-opt-out 
0  1|2023-03-15|08:45:12|Rachel|rachel@centralper...


## Core Marketing Analysis

Who are the territory managers for the sales territories assigned? What are the store IDs and cities for the stores in each assigned sales territory?

In [8]:
# I would like to find what region my assigned territory manager is responsible for
assigned_region = store_details[store_details["Territory Manager"] == given_sales_manager]["Region"].unique()
print(f"Territory Manager: {given_sales_manager} | {assigned_region[0]}\n")  # Only assigned Northeast

Territory Manager: Miami Vue | Northeast



In [9]:
# I would like to find what other territory manager exists in my region
print(f"Managers in the {assigned_region[0]} region: {store_details[store_details["Region"].isin(assigned_region)]["Territory Manager"].unique()}\n")

Managers in the Northeast region: ['Erbayne Middleton' 'Shruti Reddy' 'Bo Heap' 'Miami Vue']



In [10]:
# Finding second region to work with...
# print(f"{store_details[store_details["Territory Manager"] == "Erbayne Middleton"]["Region"].unique()}")  # Erbayne Middleton territories; Northeast only
# print(f"{store_details[store_details["Territory Manager"] == "Shruti Reddy"]["Region"].unique()}")  # Shruti Reddy territories; Northeast only
# print(f"{store_details[store_details["Territory Manager"] == "Bo Heap"]["Region"].unique()}")  # Bo Heap territories; Northeast only
# print(f"{store_details[store_details["Territory Manager"] == "Miami Vue"]["Region"].unique()}")  # Miami Vue territories; Northeast only

# assuming that each manager only has one region assigned to them... check for unique regions
print(f"List of regions and their corresponding managers:\n{store_details.groupby("Region")["Territory Manager"].unique()}\n")

List of regions and their corresponding managers:
Region
East                               [Ellen Lemon, See Ellefson]
Northeast    [Erbayne Middleton, Shruti Reddy, Bo Heap, Mia...
South          [Lana Ilana, Len Jensen, Jeff "Howdy" Richards]
West                                                [Jim Heck]
Name: Territory Manager, dtype: object



In [11]:
# Since that means I'm assigned one region, let's find another region to work with. Since we want to drive revenue up, let's see how we rank against other regions
# first, check to see how each region ranks by combining tables and clean them

# join sales table and store details table together on store ID
sales_storedetails_merge = store_sales.merge(store_details, on="Store ID", how="left")
print(sales_storedetails_merge)

# cleaning our table
is_nan = sales_storedetails_merge[sales_storedetails_merge.isna().any(axis=1)]
# print(is_nan)

# RewardsID has lots of NaN values
# is_nan[is_nan.drop(columns=["RewardsID"]).isna().any(axis=1)]  # No NaN values exist outside of RewardsID

       Transaction Date  Store ID  RewardsID   Prod Num  Sale Amount  \
0              1/1/2022       702        NaN   105349-M         8.00   
1              1/1/2022       704        NaN   105350-T       144.00   
2              1/1/2022       705        NaN   105351-M        44.00   
3              1/1/2022       705        NaN   105352-M        47.61   
4              1/1/2022       705        NaN   105353-A        20.36   
...                 ...       ...        ...        ...          ...   
335124       12/31/2025       909        NaN   105353-A        20.36   
335125       12/31/2025       909      332.0   105364-S        17.70   
335126       12/31/2025       910        NaN   105672-B        26.52   
335127       12/31/2025       910        NaN   105653-M        11.00   
335128       12/31/2025       910        NaN  105300-IT        77.81   

       Store Location     State      Territory Manager Region  \
0            Berthoud  Colorado               Jim Heck   West   
1    

In [12]:
# shorten the table view down to what we want
sales_storedetails_merge[["Store ID", "Sale Amount", "Store Location", "Territory Manager", "Region", "Region Director"]].groupby("Region")["Sale Amount"].sum().sort_values(ascending=False)

# Northeast > South > East > West; we'll compare against South since we're northeast.
assigned_region = np.append(assigned_region, "South")
print(assigned_region)

['Northeast' 'South']


In [13]:
# Let's finally answer the question.
# Territory managers for the sales region assigned:

# Northeast: Erbayne Middleton, Shruti Reddy, Bo Heap, Miami Vue (4)
print(f"{assigned_region[0]} territory managers: {store_details[store_details["Region"] == assigned_region[0]]["Territory Manager"].unique()}\n")

# South: Lana Ilana, Len Jensen, Jeff "Howdy" Richards (3)
print(f"{assigned_region[1]} territory managers: {store_details[store_details["Region"] == assigned_region[1]]["Territory Manager"].unique()}")

Northeast territory managers: ['Erbayne Middleton' 'Shruti Reddy' 'Bo Heap' 'Miami Vue']

South territory managers: ['Lana Ilana' 'Len Jensen' 'Jeff "Howdy" Richards']


In [14]:
# Store IDs of each region
# print(store_details.head())
# print(store_details.groupby(["Region", "State", "Store Location", "Territory Manager"]).count())
# print(store_details.sort_values(by=["Territory Manager", "Region", "State", "Store Location".isin(assigned_region), "Store ID"]))

# Assigned region dataframe for store_details
assigned_store_details = (
    store_details[
        store_details["Region"].isin(assigned_region)
    ]
    .groupby(
        ["Territory Manager", "Region", "State", "Store Location"]
    )
    .size()
    .reset_index(name="Count")
)

print(assigned_store_details)

# Northeast region (49)
print(f"{assigned_region[0]} store IDs: {store_details[store_details['Region'] == assigned_region[0]]['Store ID'].unique()}")
print(f"{assigned_region[0]} store count: {store_details[store_details['Region'] == assigned_region[0]]['Store ID'].count()}\n")

# Northeast region (24)
print(f"{assigned_region[1]} store IDs: {store_details[store_details['Region'] == assigned_region[1]]['Store ID'].unique()}")
print(f"{assigned_region[1]} store count: {store_details[store_details['Region'] == assigned_region[1]]['Store ID'].count()}")

   Territory Manager     Region          State       Store Location  Count
0            Bo Heap  Northeast  Massachusetts            Attleboro      1
1            Bo Heap  Northeast  Massachusetts               Boston      1
2            Bo Heap  Northeast  Massachusetts             Falmouth      1
3            Bo Heap  Northeast  Massachusetts           Framingham      1
4            Bo Heap  Northeast  Massachusetts            Haverhill      1
..               ...        ...            ...                  ...    ...
68      Shruti Reddy  Northeast       Maryland               Howard      1
69      Shruti Reddy  Northeast       Maryland        North Harford      1
70      Shruti Reddy  Northeast       Maryland            Parkville      1
71      Shruti Reddy  Northeast       Maryland  Queen Anne's County      1
72      Shruti Reddy  Northeast       Maryland              Ridgely      1

[73 rows x 5 columns]
Northeast store IDs: [818 819 820 821 822 823 731 732 733 734 735 736 737 738

In [15]:
# Cities; there is one store for each city, number matches up to number of Count ID
# Northeast region (49)
print(f"{assigned_region[0]} store IDs: {store_details[store_details['Region'] == assigned_region[0]]['Store Location'].unique()}")
print(f"{assigned_region[0]} store count: {store_details[store_details['Region'] == assigned_region[0]]['Store Location'].count()}\n")

# Northeast region (24)
print(f"{assigned_region[1]} store IDs: {store_details[store_details['Region'] == assigned_region[1]]['Store Location'].unique()}")
print(f"{assigned_region[1]} store count: {store_details[store_details['Region'] == assigned_region[1]]['Store Location'].count()}")

Northeast store IDs: ['Bangor' 'Bar Harbor' 'Kennebunkport' 'Lewiston' 'Orono' 'South Portland'
 'Annapolis' 'Back River' 'Baltimore' 'Germantown' 'Howard'
 'North Harford' 'Parkville' "Queen Anne's County" 'Ridgely' 'Boston'
 'Attleboro' 'Falmouth' 'Framingham' 'Haverhill' 'Hingham' 'Holyoke'
 'Leominster' 'Lowell' 'Lynn' 'Nantucket' 'New Bedford' 'Northampton'
 'Pittsfield' 'Provincetown' 'Quincy' 'Somerville' 'Worcester'
 'Atlantic City' 'Bayonne' 'Cape May' 'Clifton' 'East Orange' 'Hackensack'
 'Hoboken' 'Jersey City' 'Montclair' 'Morristown' 'New Brunswick' 'Newark'
 'Passaic' 'Paterson' 'Trenton' 'Vineland']
Northeast store count: 49

South store IDs: ['Cape Canaveral' 'Fort Lauderdale' 'Jacksonville' 'Key West' 'Lakeland'
 'Miami' 'Naples' 'Orlando' 'Sebring' 'Tallahassee' 'Tampa' 'Charleston'
 'Greenville' 'Arlington' 'Austin' 'Bacliff' 'Baytown' 'Beaumont'
 'Cedar Park' 'Dallas' 'Denton' 'Desoto' 'Fort Worth' 'Georgetown']
South store count: 24


What is monthly total revenue for in-store sales in each of the two sales territories, over the full period covered by the data?

In [22]:
# Monthly 
# # Northeast region (49)
# print(f"{assigned_region[0]} store IDs: {store_details[store_details['Region'] == assigned_region[0]]['Store Location'].unique()}")
# print(f"{assigned_region[0]} store count: {store_details[store_details['Region'] == assigned_region[0]]['Store Location'].count()}\n")

# # Northeast region (24)
# print(f"{assigned_region[1]} store IDs: {store_details[store_details['Region'] == assigned_region[1]]['Store Location'].unique()}")
# print(f"{assigned_region[1]} store count: {store_details[store_details['Region'] == assigned_region[1]]['Store Location'].count()}")

# monthly_sales = sales_storedetails_merge.groupby(sales_storedetails_merge['Transaction Date']
regional_sales = (
    sales_storedetails_merge[
        sales_storedetails_merge["Region"].isin(assigned_region)
    ]
    .groupby(["Transaction Date", "Region", "Store Location"])["Sale Amount"]  # trying to aggregate by date, group by region then store location
    .sum()
    .sort_values(ascending=False)
)

print(regional_sales)

Transaction Date  Region     Store Location
9/3/2025          Northeast  North Harford     13455.98
9/15/2024         Northeast  North Harford     13063.96
10/16/2023        Northeast  North Harford     13063.96
3/29/2025         Northeast  North Harford     12954.09
9/19/2025         Northeast  North Harford     12874.77
                                                 ...   
1/1/2023          South      Charleston            2.40
1/1/2022          Northeast  Passaic               2.40
8/24/2023         Northeast  Montclair             2.40
9/9/2023          Northeast  Baltimore             2.40
6/7/2022          Northeast  Somerville            2.40
Name: Sale Amount, Length: 79596, dtype: float64


How would you rank the sales performance of each store in each sales territory? Which are the top-performing stores?

Comparing the customer ID from the customer list data with the rewards ID from the sales data, who were the top customers in each sales territory?

What is the number of transactions per month by product category in each assigned territory? What is total sales revenue per month by category? What might this tell you about the most popular products, and where could there be opportunity for growth?

What is your recommendation for where to focus marketing attention in the next quarter?